# Predictive Model for Customer Loyalty

## Table of Contents
1. [Pendahuluan](#Pendahuluan)
   - Konteks Bisnis
   - Tujuan Pemodelan
   - Deskripsi Variabel
2. [EDA & Data Preparation](#EDA-&-Data-Preparation)
   - Train Label Data
   - Member Data
   - Product Data
   - Prodgram Data
   - Train Transaction Data
   - Test Transaction Data
3. [Feature Engineering](#Feature-Engineering)
4. [Modeling](#Modeling)
   - XGBoost
   - Multi-Layer Perceptron (MLP)
5. [Feature Importance](#Feature-Importance)
6. [Correlation Matrix](#Correlation-Matrix)
7. [Submission](#Submission)

In [ ]:
# =============================================================================
# Configuration
# =============================================================================
import os

# Path to dataset directory (relative to notebook location)
DATA_DIR = 'Customer Loyalty Dataset'

# Random seed for reproducibility
RANDOM_STATE = 42

# Pendahuluan

## Konteks Bisnis
Intelectra adalah sebuah perusahaan yang bergerak di bidang produksi dan penjualan susu bayi. Perusahaan ini menawarkan program keanggotaan (membership) yang memungkinkan pelanggan untuk mendapatkan poin dari setiap pembelian dan berpartisipasi dalam program loyalitas. Data yang terkumpul dari para anggota ini memberikan kesempatan bagi Intelectra untuk melakukan segmentasi pelanggan dan membangun model prediktif guna memahami perilaku belanja mereka.# 

## Tujuan Pemodelan
*Machine Learning Quest* (MLQ) yang diselenggarakan oleh INTELLECTRA ini adalah untuk membangun sebuah model klasifikasi. Secara spesifik, model ini diharapkan mampu memprediksi probabilitas seorang anggota akan melakukan pembelian kembali pada bulan berikutnya.  Model ini akan membantu perusahaan dalam merancang strategi pemasaran yang lebih tepat sasaran dan meningkatkan retensi pelanggan.

## Deskripsi Variabel

### Member
* MemberID: ID Member
* JoinDate: Tanggal terdaftar sebagai member
* DateOfBirth: Tanggal lahir member
* City: Kota tempat tinggal member
* NoOfChild: Jumlah anak yang dimiliki saat mendaftar
* YoungestKidDOB: Tanggal lahir anak yang paling muda
* EldestKidDOB: Tanggal lahir anak yang paling tua

### Transaksi
* TransactionID: ID transaksi
* MemberID: ID member untuk join ke table member
* Source: Jenis/Group dari outlet/toko member transaksi
* FK_PRODUCT_ID: ID product untuk join ke tabel product
* FK_PROD_GRAM_ID: ID product gramasi untuk join ke tabel prodgram
* Qty: Banyak kaleng/kotak yang dibeli berdasarkan jenis produk dan gramasinya
* PricePerUnit: Harga satuan produk yang dibayar oleh member pada saat transaksi, berdasarkan jenis dan gramasi produk yang dibeli
* TransactionDateTime: Tanggal transaksi yang tertera pada struk yang diinputkan member

### Produk
* ProductID: ID produk
* ProductName: Nama produk
* ProductCategory: Kategori produk
* ProductLevel: Level produk

### Prodgram
* ProdgramID: ID jenis produk
* GrammageName: Nama/besaran gramasi produk
* Point: Point dari suatu produk berdasarkan jenis dan gramasinya
* Price: Harga perkiraan berdasarkan jenis dan gramasinya

### Label
* MemberID: ID member untuk join ke tabel member
* next_buy: Status member berbelanja kembali bulan depan (1=Yes, 0=No)

## Skema Data
> *Schema Data diagram - lihat file terpisah jika tersedia*

# EDA & Data Preparation

In [ ]:
# =============================================================================
# Core Libraries
# =============================================================================
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

## Train Label Data

In [ ]:
df0 = pd.read_csv(os.path.join(DATA_DIR, 'train_label_data.csv'), sep=',')
df0.head(), df0.info()

In [ ]:
is_duplicated = df0['MemberID'].duplicated().any()
if is_duplicated:
  print("Ada nilai yang sama di kolom 'member_id'.")
else:
  print("Tidak ada nilai yang sama di kolom 'member_id'.")

In [ ]:
print(df0['next_buy'].isnull().any()), print(df0['MemberID'].isnull().any())

## Member Data

In [ ]:
df1 = pd.read_csv(os.path.join(DATA_DIR, 'member_data.csv'), sep=',')
df1.head(), df1.info()

In [ ]:
# Hitung usia sekarang anak
from datetime import date

df1['EldestKidDOB'] = pd.to_datetime(df1['EldestKidDOB'], errors='coerce')
df1['YoungestKidDOB'] = pd.to_datetime(df1['YoungestKidDOB'], errors='coerce')

df1['CurrentAgeEK'] = (pd.to_datetime( date.today() ) - df1['EldestKidDOB']).dt.days // 365
df1['CurrentAgeYK'] = (pd.to_datetime( date.today() ) - df1['YoungestKidDOB']).dt.days // 365

In [ ]:
df1 = df1.drop(columns=['DateOfBirth', 'EldestKidDOB', 'YoungestKidDOB'])

#### Drop Date0fBirth karena missing values > 30%
#### Menyederhanakan EldestKidDOB dan YoungestKidDOB ke bentuk usia sekarang (CurrentAge)

In [ ]:
print(df1['CurrentAgeEK'].value_counts().sort_index())
print('Mediannya ada di usia', df1['CurrentAgeEK'].median(),'tahun')

#### Kita asumsikan CurrentAgeEK 125 menjadi 25 dikarenakan terjadi kesalahan input data

In [ ]:
# Ganti CurrentAgeEK yang bernilai 125 dengan 25
df1.loc[df1['CurrentAgeEK'] == 125, 'CurrentAgeEK'] = 25

In [ ]:
print(df1['CurrentAgeYK'].value_counts().sort_index())
print('Mediannya ada di usia',df1['CurrentAgeYK'].median() ,'tahun')

In [ ]:
df1['JoinDate'] = pd.to_datetime(df1['JoinDate'], format='ISO8601')

In [ ]:
is_duplicated1 = df1['MemberID'].duplicated().any()
if is_duplicated1:
  print("Ada nilai yang sama di kolom 'member_id'.")
else:
  print("Tidak ada nilai yang sama di kolom 'member_id'.")

In [ ]:
df1.info(), df1.head()

## Product Data

In [ ]:
df2 = pd.read_csv(os.path.join(DATA_DIR, 'product_data.csv'), sep=',')
df2.rename(columns={'productID': 'FK_PRODUCT_ID'}, inplace=True)
df2.head(), df2.info()

## Prodgram Data

In [ ]:
df3 = pd.read_csv(os.path.join(DATA_DIR, 'prodgram_data.csv'), sep=',')
df3.rename(columns={'prodgramID': 'FK_PROD_GRAM_ID'}, inplace=True)
df3.head(), df3.info()

In [ ]:
df3.loc[df3['GrammageName'] == '1', 'GrammageName'] = np.nan

In [ ]:
df3.loc[df3['Point'] == '1', 'Point'] = np.nan

In [ ]:
def drop_rows(df3, values_to_drop=[9, 10, 15, 53, 54]):
    mask = df3['FK_PROD_GRAM_ID'].isin(values_to_drop)
    df3 = df3[~mask].copy().reset_index(drop=True)
    
    return df3

df3 = drop_rows(df3, [9, 10, 15, 53, 54])
df3.head(), df3.info()

## Train Transaction Data

In [ ]:
df4 = pd.read_csv(os.path.join(DATA_DIR, 'train_transaction_data.csv'), sep=',')
df4.head(), df4.info()

In [ ]:
is_duplicate4 = df4['TransactionID'].duplicated(keep=False)
num_of_duplicates4 = df4[is_duplicate4].shape[0]
if num_of_duplicates4 > 0:
    print(f"Ditemukan {num_of_duplicates4} baris data dengan TransactionID yang terduplikasi.\n")
    duplicated_rows4 = df4[is_duplicate4].sort_values(by='TransactionID')
    print("Berikut adalah contoh baris data dengan TransactionID yang duplikat:")
    print(duplicated_rows4)
else:
     print("Tidak ditemukan data duplikat pada kolom 'TransactionID'.")

### Deteksi Baris Duplikasi Sempurna

In [ ]:
duplicate_mask = df4.duplicated(keep=False)
exact_duplicates = df4[duplicate_mask]

print(f"Jumlah baris duplikat sempurna: {duplicate_mask.sum()}")
print(f"Jumlah baris unik yang memiliki duplikat: {len(exact_duplicates.drop_duplicates())}")

In [ ]:
if len(exact_duplicates) > 0:
    print("\n CONTOH BARIS YANG DUPLIKAT:")
    
    # Menampilkan detail beberapa contoh
    print("\nDetail contoh baris duplikat:")
    sample_duplicates = exact_duplicates
    print(sample_duplicates)
    
else:
    print("Tidak ada baris duplikat sempurna ditemukan!")

### Menghapus Duplikat Baris Sempurna

In [ ]:
df4 = df4.drop_duplicates(keep='first')  # Mempertahankan kemunculan pertama

In [ ]:
df4.duplicated().sum()

### Handling Missing Values Train

In [ ]:
df4[df4['PricePerUnit'].isna()]

In [ ]:
def fill_group(df4):
    group = df4.groupby(['Source', 'FK_PRODUCT_ID', 'FK_PROD_GRAM_ID'])['PricePerUnit'].transform('mean')
    df4['PricePerUnit'] = df4['PricePerUnit'].fillna(group)
    return df4

df4 = fill_group(df4)
df4[df4['PricePerUnit'].isna()]

In [ ]:
def fill_group2(df4):
    group2 = df4.groupby(['FK_PRODUCT_ID', 'FK_PROD_GRAM_ID'])['PricePerUnit'].transform('mean')
    df4['PricePerUnit'] = df4['PricePerUnit'].fillna(group2)
    return df4

df4 = fill_group2(df4)
df4[df4['PricePerUnit'].isna()]

In [ ]:
df4.info()

### Cek Variabel FK_PROD_GRAM_ID

In [ ]:
target_values = [9, 10, 15, 53, 54]
for value in target_values:
    exists = value in df4['FK_PROD_GRAM_ID'].values
    count = (df4['FK_PROD_GRAM_ID'] == value).sum()
    print(f"   Nilai {value}: {'✓ ADA' if exists else '✗ TIDAK ADA'} ({count} baris)")

> Dengan begitu baris 9, 10, 15, 53, dan 54 bisa di drop pada Prodgram Data

## Test Transaction Data

In [ ]:
df5 = pd.read_csv(os.path.join(DATA_DIR, 'test_transaction_data.csv'), sep=',')
df5.head(), df5.info()

In [ ]:
df5[df5['PricePerUnit'].isna()]

In [ ]:
def fill_group5(df5):
    group = df5.groupby(['Source', 'FK_PRODUCT_ID', 'FK_PROD_GRAM_ID'])['PricePerUnit'].transform('mean')
    df5['PricePerUnit'] = df5['PricePerUnit'].fillna(group)
    return df5

df5 = fill_group5(df5)
df5[df5['PricePerUnit'].isna()]

In [ ]:
def fill_group52(df5):
    group52 = df5.groupby(['FK_PRODUCT_ID', 'FK_PROD_GRAM_ID'])['PricePerUnit'].transform('mean')
    df5['PricePerUnit'] = df5['PricePerUnit'].fillna(group52)
    return df5

df5 = fill_group52(df5)
df5[df5['PricePerUnit'].isna()]

In [ ]:
df5.info()

## Merge File

### X Train

In [ ]:
X_train = pd.merge(df4, df1, on='MemberID', how='left')
X_train = pd.merge(X_train, df2, on='FK_PRODUCT_ID', how='left')
X_train = pd.merge(X_train, df3, on='FK_PROD_GRAM_ID', how='left')
X_train.info(), X_train.head()

In [ ]:
for col in X_train.columns:
    print(col)
    print(X_train[col].unique())
    print("")

### X Test

In [ ]:
X_test = pd.merge(df5, df1, on='MemberID', how='left')
X_test = pd.merge(X_test, df2, on='FK_PRODUCT_ID', how='left')
X_test = pd.merge(X_test, df3, on='FK_PROD_GRAM_ID', how='left')
X_test.info(), X_test.head()

In [ ]:
for col in X_test.columns:
    print(col)
    print(X_test[col].unique())
    print("")

### y Train

In [ ]:
y_train = df0
y_train.info(), y_train.head()

# Feature Engineering

In [ ]:
# Time-Based Features
X_train['TransactionDatetime'] = pd.to_datetime(X_train['TransactionDatetime'])
X_test['TransactionDatetime'] = pd.to_datetime(X_test['TransactionDatetime'])

X_train['transaction_dayofweek'] = X_train['TransactionDatetime'].dt.dayofweek  # Senin=0, Minggu=6
X_train['TransactionMonth'] = X_train['TransactionDatetime'].dt.month
X_train['is_weekend'] = (X_train['transaction_dayofweek'] >= 5).astype(int)
X_train['KidGap'] = X_train['CurrentAgeEK'] - X_train['CurrentAgeYK']
X_test['transaction_dayofweek'] = X_test['TransactionDatetime'].dt.dayofweek
X_test['TransactionMonth'] = X_test['TransactionDatetime'].dt.month
X_test['is_weekend'] = (X_test['transaction_dayofweek'] >= 5).astype(int)
X_test['KidGap'] = X_test['CurrentAgeEK'] - X_test['CurrentAgeYK']

In [ ]:
for col in ['JoinDate']:
    X_train[f'{col}_year'] = X_train[col].dt.year
    X_train[f'{col}_month'] = X_train[col].dt.month
    X_train[f'{col}_day'] = X_train[col].dt.day
    X_test[f'{col}_year'] = X_test[col].dt.year
    X_test[f'{col}_month'] = X_test[col].dt.month
    X_test[f'{col}_day'] = X_test[col].dt.day

In [ ]:
X_train = X_train.drop(columns=['JoinDate','TransactionDatetime','GrammageName','Price'])
X_test = X_test.drop(columns=['JoinDate','TransactionDatetime','GrammageName','Price'])

In [ ]:
X_train

# Categorical Handling

#### Categorical Handling dilakukan terpisah menurut skala pengukuran pada setiap variabel

In [ ]:
X_train.replace({"ProductCategory": {'A': 0, 'B':1}}, inplace=True)
X_train.replace({"ProductLevel": {'Original': 0, 'Premium':1}}, inplace=True)
X_test.replace({"ProductCategory": {'A': 0, 'B':1}}, inplace=True)
X_test.replace({"ProductLevel": {'Original': 0, 'Premium':1}}, inplace=True)

## Frequency Encoding

In [ ]:
city_map = X_train['City'].value_counts().to_dict()
X_train['City_encoded'] = X_train['City'].map(city_map)
source_map = X_train['Source'].value_counts().to_dict()
X_train['Source_encoded'] = X_train['Source'].map(source_map)
city_map2 = X_test['City'].value_counts().to_dict()
X_test['City_encoded'] = X_test['City'].map(city_map2)
source_map2 = X_test['Source'].value_counts().to_dict()
X_test['Source_encoded'] = X_test['Source'].map(source_map2)

## One-Hot Encoding

In [ ]:
X_train = pd.get_dummies(X_train, columns=['ProductName'], prefix='Product')
X_test = pd.get_dummies(X_test, columns=['ProductName'], prefix='Product')

In [ ]:
X_train = X_train.drop(columns=['City','Source'])
X_test = X_test.drop(columns=['City','Source'])

In [ ]:
for col in X_train.columns:
    print(col)
    print(X_train[col].unique())
    print("")

# Modelling

In [ ]:
from datetime import datetime
import warnings
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, GridSearchCV
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Merge dengan label
X_train_final = pd.merge(X_train, y_train, on='MemberID', how='inner')

# Siapkan X dan y
X_train = X_train_final.drop(columns=['next_buy'])
y_train = X_train_final['next_buy']

## XGBoost

In [ ]:
X_train_cleaned = X_train.drop(columns=['TransactionID','MemberID'])
X_test_cleaned = X_test.drop(columns=['TransactionID','MemberID'])

In [ ]:
import xgboost as xgb
xgb_model = xgb.XGBClassifier(scale_pos_weight=15.41509, random_state=42)
xgb_model.fit(X_train_cleaned, y_train)
y_pred_xgb = xgb_model.predict(X_test_cleaned[X_train_cleaned.columns]) # pastikan urutan kolom sama

In [ ]:
pred_xgb = pd.DataFrame({
    'MemberID': X_test['MemberID'],
    'next_buy': y_pred_xgb
})

sample = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
submission_xgb = sample.copy()
pred_dict_xgb = dict(zip(pred_xgb['MemberID'], pred_xgb['next_buy']))
submission_xgb['next_buy'] = submission_xgb['MemberID'].map(pred_dict_xgb)
submission_xgb['next_buy'] = submission_xgb['next_buy'].astype(int)
submission_xgb.to_csv('submission_xgb.csv', index=False)
submission_xgb.head(), submission_xgb.info()

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
# Calculate permutation feature importance
perm_importance = permutation_importance(xgb_model, X_train_cleaned, y_train, n_repeats=10, random_state=42)

# Plot permutation feature importance
sorted_idx = perm_importance.importances_mean.argsort()
plt.figure(figsize=(10, 6))
plt.barh(range(len(sorted_idx)), perm_importance.importances_mean[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), [f'feature_{i}' for i in sorted_idx])
plt.xlabel('Permutation Feature Importance')
plt.ylabel('Feature')
plt.title('Permutation Feature Importance (XGBoost)')
plt.tight_layout()
plt.show()

### Hyperparameter & Cross-validation XGB

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score, classification_report

In [ ]:
# Split data untuk validasi
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_cleaned, y_train, 
    test_size=0.2, 
    stratify=y_train, 
    random_state=42
)

# Hitung rasio untuk scale_pos_weight
neg_count = np.sum(y_tr == 0)
pos_count = np.sum(y_tr == 1)
scale_pos_weight_value = neg_count / pos_count

print(f"Jumlah kelas Negatif: {neg_count}")
print(f"Jumlah kelas Positif: {pos_count}")
print(f"Nilai scale_pos_weight: {scale_pos_weight_value:.2f}")

In [ ]:
# Model XGBoost dengan hyperparameter yang lebih baik
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=2,
    scale_pos_weight=4.62,
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)

In [ ]:
# Training dengan early stopping
xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    early_stopping_rounds=50,
    verbose=100
)

In [ ]:
# Evaluasi di validation set
y_val_pred = xgb_model.predict(X_val)
val_accuracy = balanced_accuracy_score(y_val, y_val_pred)
print(f"Validation Balanced Accuracy: {val_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

In [ ]:
# Cross-validation untuk memastikan generalisasi
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    xgb_model, X_train_cleaned, y_train, 
    cv=skf, 
    scoring='balanced_accuracy'
)
print(f"\nCV Balanced Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

In [ ]:
neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)
scale_pos_weight_value = neg_count / pos_count

print(f"Jumlah kelas Negatif: {neg_count}")
print(f"Jumlah kelas Positif: {pos_count}")
print(f"Nilai scale_pos_weight: {scale_pos_weight_value:.2f}")

In [ ]:
# Fit final model dengan semua training data
final_model = xgb.XGBClassifier(
    n_estimators=xgb_model.best_iteration if hasattr(xgb_model, 'best_iteration') else 1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=2,
    scale_pos_weight=4.62,
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)

final_model.fit(X_train_cleaned, y_train)

In [ ]:
# Prediksi untuk submission
y_pred_xgb = final_model.predict(X_test_cleaned[X_train_cleaned.columns])
pred_xgb = pd.DataFrame({
    'MemberID': X_test['MemberID'],
    'next_buy': y_pred_xgb
})

submission_xgb = sample.copy()
pred_dict_xgb = dict(zip(pred_xgb['MemberID'], pred_xgb['next_buy']))
submission_xgb['next_buy'] = submission_xgb['MemberID'].map(pred_dict_xgb)
submission_xgb['next_buy'] = submission_xgb['next_buy'].astype(int)

print(f"\nPrediction distribution:")
print(submission_xgb['next_buy'].value_counts(normalize=True))

submission_xgb.to_csv('submission_xgb_improved.csv', index=False)
print(f"\nSubmission saved as 'submission_xgb_improved.csv'")

# Feature Importance

In [ ]:
# Cek distribusi label
print("Train label distribution:")
print(y_train.value_counts(normalize=True))

# Cek missing values
print("\nMissing values in features:")
print(X_train_cleaned.isnull().sum().sum())

# Cek feature importance
feature_importance = pd.DataFrame({
    'feature': X_train_cleaned.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop important features:")
print(feature_importance)

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
# Calculate permutation feature importance
perm_importance = permutation_importance(xgb_model, X_train_cleaned, y_train, n_repeats=10, random_state=42)

# Plot permutation feature importance
sorted_idx = perm_importance.importances_mean.argsort()
plt.figure(figsize=(10, 6))
plt.barh(range(len(sorted_idx)), perm_importance.importances_mean[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), [f'feature_{i}' for i in sorted_idx])
plt.xlabel('Permutation Feature Importance')
plt.ylabel('Feature')
plt.title('Permutation Feature Importance (XGBoost)')
plt.tight_layout()
plt.show()

In [ ]:
perm_importance2 = permutation_importance(final_model, X_train_cleaned, y_train, n_repeats=10, random_state=42)

# Plot permutation feature importance
sorted_idx2 = perm_importance2.importances_mean.argsort()
plt.figure(figsize=(10, 6))
plt.barh(range(len(sorted_idx2)), perm_importance2.importances_mean[sorted_idx2], align='center')
plt.yticks(range(len(sorted_idx2)), [f'feature_{i}' for i in sorted_idx2])
plt.xlabel('Permutation Feature Importance')
plt.ylabel('Feature')
plt.title('Permutation Feature Importance (XGBoost)')
plt.tight_layout()
plt.show()

In [ ]:
import eli5
from eli5.sklearn import PermutationImportance

perm = PermutationImportance(xgb_model, random_state=42).fit(X_train_cleaned, y_train)
eli5.show_weights(perm, feature_names = X_train_cleaned.columns.tolist())

# Corelation Matrix

In [ ]:
import seaborn as sns

In [ ]:
# Korelasi antar fitur (X_train_cleaned)
correlation_matrix = X_train_cleaned.corr()

# Plot heatmap korelasi antar fitur
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, 
            annot=True, 
            cmap='coolwarm', 
            center=0,
            square=True,
            fmt='.2f',
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix - Features')
plt.tight_layout()
plt.show()

In [ ]:
# Korelasi fitur dengan target variable
# Hitung korelasi setiap fitur dengan target (y_train)
feature_target_corr = X_train_cleaned.corrwith(y_train).sort_values(key=abs, ascending=False)
print("Korelasi fitur dengan target (diurutkan berdasarkan nilai absolut):")
print(feature_target_corr)

# Plot korelasi fitur dengan target
plt.figure(figsize=(10, 8))
feature_target_corr.plot(kind='barh', color=['red' if x < 0 else 'blue' for x in feature_target_corr])
plt.title('Correlation between Features and Target Variable')
plt.xlabel('Correlation Coefficient')
plt.ylabel('Features')
plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Multi-Layer Perceptron (MLP)

In [ ]:
pip install keras-tuner

In [ ]:
import tensorflow as tf
import keras_tuner as kt
from sklearn.utils import class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
# Hitung bobot kelas
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = dict(enumerate(class_weights))

print(f"Class Weights: {class_weights_dict}")

In [ ]:
# Membangun model
def build_model(hp):
    """Fungsi untuk membangun model yang bisa dituning."""
    model = Sequential()

    # Tentukan jumlah neuron di layer pertama
    hp_units_1 = hp.Int('units_1', min_value=64, max_value=512, step=64)
    model.add(Dense(units=hp_units_1, activation='relu', input_shape=(X_train_cleaned.shape[1],)))
    model.add(BatchNormalization())
    # Tentukan dropout rate
    hp_dropout_1 = hp.Float('dropout_1', min_value=0.2, max_value=0.5, step=0.1)
    model.add(Dropout(rate=hp_dropout_1))

    # Tentukan jumlah neuron di layer kedua
    hp_units_2 = hp.Int('units_2', min_value=32, max_value=256, step=32)
    model.add(Dense(units=hp_units_2, activation='relu'))
    model.add(BatchNormalization())
    hp_dropout_2 = hp.Float('dropout_2', min_value=0.2, max_value=0.5, step=0.1)
    model.add(Dropout(rate=hp_dropout_2))

    model.add(Dense(1, activation='sigmoid'))

    # Tentukan learning rate untuk optimizer Adam
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc')] # Fokus pada AUC sebagai metrik utama tuning
    )
    return model

# Inisialisasi Tuner
# Hyperband adalah algoritma tuning yang efisien
tuner = kt.Hyperband(
    build_model,
    objective=kt.Objective('val_auc', direction='max'), # Tujuan: maksimalkan val_auc
    max_epochs=50,
    factor=3,
    directory='keras_tuner_dir',
    project_name='next_buy_prediction'
)

# Siapkan callback untuk menghentikan pencarian lebih awal
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

# Mulai pencarian hyperparameter
# Gunakan data dan class weights yang sama seperti sebelumnya
tuner.search(
    X_train_cleaned,
    y_train,
    epochs=50,
    validation_split=0.2,
    class_weight=class_weights_dict, # Tetap gunakan class_weight!
    callbacks=[stop_early]
)

# Dapatkan hyperparameter terbaik
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
Hyperparameter terbaik yang ditemukan:
- Unit Layer 1: {best_hps.get('units_1')}
- Dropout Layer 1: {best_hps.get('dropout_1')}
- Unit Layer 2: {best_hps.get('units_2')}
- Dropout Layer 2: {best_hps.get('dropout_2')}
- Learning Rate: {best_hps.get('learning_rate')}
""")

In [ ]:
# Inisialisasi Stratified K-Fold
N_SPLITS = 5 # Bisa 5 atau 10, tergantung ukuran data
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Siapkan array untuk menyimpan prediksi pada data tes
test_predictions_ensemble = np.zeros(X_test_cleaned.shape[0])

# Siapkan list untuk menyimpan skor validasi dari setiap fold
fold_val_scores = []

# Loop untuk setiap fold
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_cleaned, y_train)):
    print(f"===== FOLD {fold+1} =====")
    
    # Pisahkan data untuk fold ini - GUNAKAN .iloc untuk row indexing
    X_train_fold, X_val_fold = X_train_cleaned.iloc[train_idx], X_train_cleaned.iloc[val_idx]
    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Bangun model dengan hyperparameter terbaik
    # PENTING: model harus di-build ulang di setiap fold!
    model_cv = tuner.hypermodel.build(best_hps)
    
    # Latih model pada data fold ini
    history_cv = model_cv.fit(
        X_train_fold,
        y_train_fold,
        epochs=100,
        batch_size=64,
        validation_data=(X_val_fold, y_val_fold),
        class_weight=class_weights_dict,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=10, restore_best_weights=True)
        ],
        verbose=0 # Matikan log per epoch agar tidak terlalu ramai
    )
    
    # Evaluasi & simpan skor AUC pada validation set
    val_auc = max(history_cv.history['val_auc'])
    fold_val_scores.append(val_auc)
    print(f"Validation AUC for Fold {fold+1}: {val_auc:.4f}")
    
    # Lakukan prediksi pada data tes dan tambahkan ke array ensemble
    test_pred_fold = model_cv.predict(X_test_cleaned).flatten()
    test_predictions_ensemble += test_pred_fold

# Rata-ratakan prediksi dari semua fold
final_predictions_prob = test_predictions_ensemble / N_SPLITS
final_predictions_class = (final_predictions_prob > 0.5).astype(int)

print(f"\nAverage Validation AUC across all folds: {np.mean(fold_val_scores):.4f} (+/- {np.std(fold_val_scores):.4f})")

# Optional: Tampilkan distribusi prediksi probabilitas
print(f"\nPrediction probability distribution:")
print(f"Min: {final_predictions_prob.min():.4f}")
print(f"Max: {final_predictions_prob.max():.4f}")
print(f"Mean: {final_predictions_prob.mean():.4f}")
print(f"Std: {final_predictions_prob.std():.4f}")

# Optional: Tampilkan proporsi prediksi kelas
print(f"\nClass distribution in predictions:")
print(f"Class 0: {np.sum(final_predictions_class == 0)} ({np.mean(final_predictions_class == 0)*100:.1f}%)")
print(f"Class 1: {np.sum(final_predictions_class == 1)} ({np.mean(final_predictions_class == 1)*100:.1f}%)")

In [ ]:
# Buat file submission
submission = pd.DataFrame({'MemberID': X_test['MemberID'], 'next_buy': final_predictions_prob.flatten()})

final_submission_df = submission.groupby('MemberID').agg(
    final_probability=('next_buy', 'mean')
).reset_index()

# Konversi probabilitas akhir menjadi kelas (0 atau 1)
final_submission_df['next_buy'] = (final_submission_df['final_probability'] > 0.5).astype(int) # threshold

submission = final_submission_df[['MemberID', 'next_buy']]
submission.to_csv('submission.csv', index=False)
submission.info(), submission.head()